# 4.3 Source, Receiver, And Surface Gradings

Gradings request additional local refinement near acquisition points or named model surfaces. They are useful when a source, receiver cable, or interface needs controlled resolution beyond the global wavelength target.

By the end, you should be able to use source, receiver, and surface gradings as local refinement requests and verify their effect in ParaView output.


## How To Read This Tutorial

Gradings are geometric refinement requests tied to meaningful objects: sources, receivers, and surfaces. They complement material/adaptivity fields by saying where the numerical solution needs extra local resolution because of acquisition or geometry.

The notebook shows the pattern used in production: name the relevant model surface, author source/receiver/surface gradings, run a ParaView QC job, and save screenshots that reveal whether refinement landed where intended.

## Design Notes

Gradings add local mesh-size intent around geometric features. They complement material adaptivity fields: `hmin`, `hmax`, and `epw_mult` live on the model, while gradings are attached to sources, receivers, or named surfaces.

| Grading | API pattern | What it protects |
| --- | --- | --- |
| Source grading | `sim.mesh.set_source_grading(...)` | Near-source singular behavior and wavefield curvature. |
| Receiver grading | `sim.mesh.set_receiver_grading(...)` | Receiver interpolation accuracy and local response. |
| Surface grading | `sim.mesh.add_surface_grading("interface", ...)` | Interfaces, topography, contacts, and other named surfaces. |

The distances `d0` and `d1` define the inner and outer grading bands; `factor` sets the strength at the feature and `power` curves the transition back to the background size. The default `power=1` is linear; larger values keep stronger refinement closer to the feature before relaxing. `factor` and `power` may also be dictionaries keyed by the active global coordinate-system axis names, for example `{"offset": 2.0, "depth": 1.5}`. Start conservative, inspect the ParaView mesh, then tighten only where the result needs it.


## Imports

The examples use the public `import frequensolve as fs` API plus standard scientific Python tools for inspection and plotting. Keeping imports ordinary makes the notebook easier to reuse in analysis or operations notebooks.

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Image, display
import frequensolve as fs

u = fs.ureg


## Model With A Named Interface

The named `interface` surface becomes a target for surface grading. Clear surface names matter because mesh controls refer to them by name, and those names also appear in exported project artifacts and ParaView outputs.

The layer sequence still determines material interfaces. The grading does not create a new layer or alter the physics; it only asks the mesher to pay extra attention near an already-named geometric feature.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="mesh_gradings",
    path="./scratch/tutorials/mesh_gradings",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="mesh_gradings",
    physics="acoustic",
    dimension=2,
    units={
        "length": "km",
        "velocity": "km/s",
        "density": "g/cm^3",
    },
)

model = fs.LayeredModel(
    name="model",
    dimension=2,
    x_limits=[0.0, 1.0],
)
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
model.add_surface(name="interface", depth=0.25 * u.km)
model.add_layer(name="basement", properties={"Vp": 2.5 * u.km / u.s, "Rho": 2.2 * u.g / u.cm**3})
model.add_surface(name="bottom", depth=0.5 * u.km)
sim += model
model.plot("vp", figsize=(7, 3), aspect="equal")


## Grading Controls And ParaView Output

Source and receiver gradings follow acquisition geometry. Surface gradings follow model geometry. Combining all three in one small run makes the ownership clear: acquisition-driven refinement comes from sources and receivers, while geology-driven refinement comes from named surfaces.

The ParaView output includes the pressure field and subdomain labels. After a successful run, inspect the mesh edges around the source, receiver line, and interface to confirm that each grading affected the expected region and did not over-refine the entire domain.


In [ ]:
sim += model.hex_mesh_generator([4, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=30.0)
sim.mesh.set_source_grading(d0=0.01, d1=0.08, factor=2.0, power=2.0)
sim.mesh.set_receiver_grading(d0=0.01, d1=0.05, factor=1.5)
sim.mesh.add_surface_grading(
    "interface",
    d0=0.0,
    d1=0.04,
    factor=2.0,
    mode="abs_band",
)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(
    conditions=["pml"],
    boundaries=["x_min", "x_max", "z_max"],
    pml_wavelengths=0.5,
)

acq = fs.Acquisition()
acq.add_sources(kind="scalar", coords=[[0.5, 0.05]])
node = fs.ReceiverNode(name="hydrophone")
node.add_component(name="p", field="pressure")
acq.add_receiver_group(
    name="surface",
    device=node,
    coords=[[x, 0.025] for x in np.linspace(0.1, 0.9, 21)],
)
sim += acq
sim += fs.Discretization()

site = fs.Site()
job = fs.FrequencyDomainJob(
    name="freq_gradings",
    simulation=sim,
    f_list=[25.0],
    outputs=[
        fs.VtkOutput.domain(
            name="gradings",
            fields=["pressure"],
            properties=["vp", "Subdomain"],
            show_pml=True,
            upscale=1,
            order=2,
        )
    ],
)
result = site.submit(job).wait()


## Discover ParaView Files

The result object records files produced by the job. Filtering by base name and suffix keeps the notebook independent of the exact project directory layout.


In [ ]:
vtu_files = result.output_files(base="gradings", suffix=".vtu", existing=True)
[str(path) for path in vtu_files]


## Render Persistent Mesh Screenshots

These cells use the generated VTK output as evidence for the grading controls. The screenshots are written to `assets/` so the notebook remains useful after the PyVista process is gone.

When reviewing the images, look for local refinement bands rather than global refinement. A good grading should be visible near the feature it protects and should relax away from that feature according to the chosen `d0`, `d1`, `factor`, and `power` values.


In [ ]:
image_dir = Path("./assets")
image_dir.mkdir(exist_ok=True)
rendered = []
for field, filename in [('vp', 'gradings_vp_edges.png'), ('pressure', 'gradings_pressure.png')]:
    screenshot = image_dir / filename
    fs.plot_vtu(
        vtu_files[0],
        field=field,
        show_edges=True,
        scalar_bar=True,
        show=False,
        screenshot=screenshot,
        window_size=(1100, 500),
    )
    rendered.append(screenshot)

for screenshot in rendered:
    display(Image(filename=str(screenshot)))


## Before Moving On

Use gradings when refinement should follow modeling intent rather than a uniform global target. Sources and receivers often need local resolution because they are singular or highly localized; interfaces may need resolution because they control scattering and mode conversion.

The mesh screenshot is the review artifact. If refinement is missing, check names and grading distances before increasing global resolution.

## Result Review Checklist

Gradings are local requests, so the final review should focus on locality. The mesh should tighten near the source, receiver line, and named interface, then relax away from those features.

| Feature | Expected visual cue |
| --- | --- |
| Source grading | Smaller elements close to the source coordinate, with gradual relaxation by `d1`. |
| Receiver grading | A refinement band following the receiver line, not the whole top layer. |
| Surface grading | A band around `interface`; changing the surface name should change where the band appears. |
| PML | PML cells are visible when `show_pml=True`, but grading should not be interpreted as a PML thickness control. |

When tuning real models, start with one grading at a time and add the next only after the first is visible in a ParaView edge view. That keeps cost changes attributable.
